In [1]:
import torch
import torch.nn as nn
import numpy as np
from time import time
from models import get_model
import os
import random
from dataset import get_data
from dataset.data_deal import Simu_Dataset
from environment import Env
from time import time
from torch.utils.data import WeightedRandomSampler
import matplotlib.pyplot as plt
from agent import Agent
from inference import Inference
from environment import Env
import seaborn as sns
from math import ceil
from tqdm import tqdm
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
class arg:
    def __init__(self) -> None:
        pass

In [3]:
args = arg()
args.random_seed = int(17)
args.model = 'simple'
args.dataset = 'ACIC2016'
args.task_id = int(2)
args.data_type = str('dependent_dependent_complex')

In [4]:
args.train_lr = 0.003
args.val_test_split = [0.25,0.25]
args.n_feature = int(58) if args.dataset == 'ACIC2016' else int(25)
args.missing_ratio = float(1)
args.disable_cuda = False
args.complete = False
args.pretrain = int(1000)
args.pretrain_sample = str('both')
args.mode = str('double')
args.decay = float(0.999)
args.gamma = float(1)
args.dropout = False
args.batchnorm = False
args.done_action_train = False
args.p = float(0)
args.group_norm = float(0)
args.save_dir = str('result')
args.inf_hidden_sizes = [512,512]
args.policy_hidden_sizes = [32]
args.shared_dim = int(16)
args.target_update_freq = int(100)
args.eps_start = float(1.)
args.eps_end = float(0.1)
args.decay_rate = float(2)
args.n_env = int(32)
args.nsteps = int(4)
args.normalize = True
args.embedded_dim = int(16)
args.lstm_size = int(16)
args.n_shuffle = int(5)
args.r_cost = float(1.0)
args.cost_from_file = False
args.batch_size = int(128)
args.message = str('')
args.buffer_size = int(10000)
random.seed(args.random_seed)
np.random.seed(args.random_seed)
torch.manual_seed(args.random_seed)
if not args.disable_cuda and torch.cuda.is_available():
    args.device = torch.device('cuda')
    torch.cuda.manual_seed(args.random_seed)
else:
    args.device = torch.device('cpu')
args.save_dir = os.path.join(os.getcwd(),args.save_dir)
if args.dataset == 'simu_data':
    args.save_path = args.dataset + '_' + args.data_type
elif args.dataset == 'ACIC2016':
    args.save_path = args.dataset + '_' + str(args.task_id)
else:
    args.save_path = args.dataset
args.save_path = args.save_path + '_cost{}'.format(args.r_cost)
args.save_path = os.path.join(args.save_dir, args.save_path)
args.csv_path = args.save_path
args.save_path = args.save_path + '_seed{}'.format(args.random_seed)
args.data_path = os.path.join(os.getcwd(),'dataset')
if not os.path.exists(args.save_path):
    os.makedirs(args.save_path)


In [5]:
args.X_mode,args.T_mode,args.Y_mode = args.data_type.split('_')
traindata,testdata,valdata = get_data(args)

In [6]:
model = get_model(args)
inf = Inference(model,'T_mode',args,5000)
agent = Agent(model,args,5000)
train_env = Env(args.n_env,traindata,model,args.r_cost)
val_env = Env(args.n_env,valdata,model,args.r_cost)
test_env = Env(args.n_env,testdata,model,args.r_cost)
args.missing_ratio = float(0)
inf.pretrain(traindata,valdata,args,25000,128)

start_pretrain
epoch: 10  train_loss: 631.2340698242188  val_loss: 867.9619750976562
epoch: 20  train_loss: 597.7649536132812  val_loss: 771.3619995117188
epoch: 30  train_loss: 342.5731201171875  val_loss: 228.91456604003906
epoch: 40  train_loss: 87.19337463378906  val_loss: 115.73643493652344
epoch: 90  train_loss: 137.57395935058594  val_loss: 32.47846603393555
epoch: 190  train_loss: 12.99499797821045  val_loss: 22.432771682739258
epoch: 230  train_loss: 18.34807586669922  val_loss: 19.145090103149414
epoch: 250  train_loss: 9.535323143005371  val_loss: 19.007675170898438
epoch: 300  train_loss: 17.247562408447266  val_loss: 18.401142120361328
epoch: 390  train_loss: 9.550514221191406  val_loss: 14.38012409210205
epoch: 640  train_loss: 4.573049545288086  val_loss: 10.727920532226562
epoch: 830  train_loss: 6.507111072540283  val_loss: 9.345693588256836
epoch: 920  train_loss: 4.276041507720947  val_loss: 9.074462890625
epoch: 1110  train_loss: 6.947517395019531  val_loss: 9.03498

In [7]:
for missingratio in np.arange(0.9,-0.01,-0.1):
    mask = torch.rand_like(testdata.features)
    mask = (mask>missingratio).int()
    incomplete_testdata = Simu_Dataset(testdata.features,testdata.treatments,testdata.y_fact,testdata.y_cf,testdata.mu)
    incomplete_testdata.features = incomplete_testdata.features.masked_fill(~mask.bool(),torch.nan)
    features_mean = incomplete_testdata.features.nanmean(0)
    incomplete_testdata.features = torch.where(~mask.bool(),features_mean,incomplete_testdata.features)
    print("missingratio:",missingratio)
    inf.test(incomplete_testdata,args)
    print('\n')


missingratio: 0.9
start_inference_test
finish_inference_test
time_use: 0.024008750915527344
mse of tau: tensor(63.5842, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(122.3187, device='cuda:0', grad_fn=<DivBackward0>)


missingratio: 0.8
start_inference_test
finish_inference_test
time_use: 0.03045487403869629
mse of tau: tensor(62.2853, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(109.2957, device='cuda:0', grad_fn=<DivBackward0>)


missingratio: 0.7000000000000001
start_inference_test
finish_inference_test
time_use: 0.026302337646484375
mse of tau: tensor(49.3454, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(87.1024, device='cuda:0', grad_fn=<DivBackward0>)


missingratio: 0.6000000000000001
start_inference_test
finish_inference_test
time_use: 0.023783445358276367
mse of tau: tensor(50.0955, device='cuda:0', grad_fn=<DivBackward0>)
mse of y_fact: tensor(86.6516, device='cuda:0', grad_fn=<DivBackward0>)


missingratio: 0.5000000000000